In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [21]:
#train test split
def split_scaler(indep_X, dep_Y):
    #from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size=0.25, random_state=0)
    
    # to standardise the data
    from sklearn.preprocessing import StandardScaler
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)
    
    #save the sacled model for delpyment stage.
    #import joblib
    #joblib.dump(sc, "scaler.pkl")
    
    return X_train, X_test, y_train, y_test

In [22]:
#Create a function for RFE feature selection with each algorithms.
def rfeFeature(indep_X, dep_Y, n):
    rfe_feature_list = []
    rfe_selected_names = []
    
    #1. Liner regression
    from sklearn.linear_model import LinearRegression
    lin = LinearRegression()
    
    #2. Support Vector liner
    from sklearn.svm import SVR
    SVRl = SVR(kernel = 'linear')
        
    #3. Decision Tree
    from sklearn.tree import DecisionTreeRegressor
    dt = DecisionTreeRegressor(random_state = 0)
    
    #4. Random Forest
    from sklearn.ensemble import RandomForestRegressor
    rf = RandomForestRegressor(n_estimators = 10, random_state = 0)
    
    #store the above RFE models in list below.
    rfe_model_list = [lin, SVRl, dt, rf]
    
    #loop the data set through each RFE model for feature selection
    for i in rfe_model_list:
        from sklearn.feature_selection import RFE
        rferr = RFE(i,n)
        fit1 = rferr.fit(indep_X, dep_Y) #to select features from input as per output
        
        #store the selecetd feat column names in a list
        mask = fit1.support_
        selected_cols = indep_X.columns[mask]
        rfe_selected_names.append(selected_cols)
        
        #store the selected features in list
        selected_feature = fit1.transform(indep_X) # to store only selected features
        rfe_feature_list.append(selected_feature)
        
    return rfe_feature_list, rfe_selected_names    

In [23]:
#metrics for regression

def r2_prediction(regressor,X_test,y_test):
    y_pred = regressor.predict(X_test)
    from sklearn.metrics import r2_score
    r2=r2_score(y_test,y_pred)
    return r2

In [24]:
# create models for each regression algorithm

def Linear(X_train,y_train,X_test):       
        from sklearn.linear_model import LinearRegression
        regressor = LinearRegression()
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2   
    
def svm_linear(X_train,y_train,X_test):
                
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'linear')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
    
def svm_NL(X_train,y_train,X_test):
                
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'rbf')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2      

def Decision(X_train,y_train,X_test):
        
        # Fitting K-NN to the Training setC
        from sklearn.tree import DecisionTreeRegressor
        regressor = DecisionTreeRegressor(random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2       

def random(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.ensemble import RandomForestRegressor
        regressor = RandomForestRegressor(n_estimators = 10, random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2

In [25]:
#store r2 value in a dataframe
def rfe_regression(r2_linear, r2_svm_linear, r2_svm_NL, r2_Decision, r2_random):
    rfedataframe = pd.DataFrame(index=['Linear', 'SVM', 'DT', 'RF'],columns=['Linear','SVML','SVMNL','DT','RF'])
    for number, idex in enumerate(rfedataframe.index):
        rfedataframe['Linear'][idex] = r2_linear[number]
        rfedataframe['SVML'][idex] = r2_svm_linear[number]
        rfedataframe['SVMNL'][idex] = r2_svm_NL[number]
        rfedataframe['DT'][idex] = r2_Decision[number]
        rfedataframe['RF'][idex] = r2_random[number]
    return rfedataframe

In [26]:
dataset = pd.read_csv("50_Startups.csv")
dataset = pd.get_dummies(dataset, drop_first=True)
dataset = dataset.astype(int)

In [27]:
dataset

,R&D Spend,Administration,Marketing Spend,Profit,State_Florida,State_New York
0,165349,136897,471784,192261,0,1
1,162597,151377,443898,191792,0,0
2,153441,101145,407934,191050,1,0
3,144372,118671,383199,182901,0,1
4,142107,91391,366168,166187,1,0
5,131876,99814,362861,156991,0,1
6,134615,147198,127716,156122,0,0
7,130298,145530,323876,155752,1,0
8,120542,148718,311613,152211,0,1
9,123334,108679,304981,149759,0,0


In [28]:
indep_X = dataset.drop('Profit', 1) # Dropping o/p
dep_Y = dataset['Profit'] #keep only op

C:\Anaconda3\envs\aiml\lib\site-packages\ipykernel_launcher.py:1: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only
  """Entry point for launching an IPython kernel.


In [36]:
rfe_feature_list,rfe_selected_names = rfeFeature(indep_X, dep_Y, 3)

C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\feature_selection\rfe.py:167: DeprecationWarning: `np.bool` is a deprecated alias for the builtin `bool`. To silence this warning, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  support_ = np.ones(n_features, dtype=np.bool)
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\feature_selection\rfe.py:168: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance

In [37]:
rfe_feature_list

[array([[165349,      0,      1],
        [162597,      0,      0],
        [153441,      1,      0],
        [144372,      0,      1],
        [142107,      1,      0],
        [131876,      0,      1],
        [134615,      0,      0],
        [130298,      1,      0],
        [120542,      0,      1],
        [123334,      0,      0],
        [101913,      1,      0],
        [100671,      0,      0],
        [ 93863,      1,      0],
        [ 91992,      0,      0],
        [119943,      1,      0],
        [114523,      0,      1],
        [ 78013,      0,      0],
        [ 94657,      0,      1],
        [ 91749,      1,      0],
        [ 86419,      0,      1],
        [ 76253,      0,      0],
        [ 78389,      0,      1],
        [ 73994,      1,      0],
        [ 67532,      1,      0],
        [ 77044,      0,      1],
        [ 64664,      0,      0],
        [ 75328,      1,      0],
        [ 72107,      0,      1],
        [ 66051,      1,      0],
        [ 6560

In [38]:
rfe_selected_names

[Index(['R&D Spend', 'State_Florida', 'State_New York'], dtype='object'),
 Index(['R&D Spend', 'State_Florida', 'State_New York'], dtype='object'),
 Index(['R&D Spend', 'Administration', 'Marketing Spend'], dtype='object'),
 Index(['R&D Spend', 'Administration', 'Marketing Spend'], dtype='object')]

In [39]:
#create empty list to store r2 values

r2_linear = []
r2_svm_linear = [] 
r2_svm_NL = [] 
r2_Decision = []
r2_random =[]

In [40]:
for i in rfe_feature_list:# loop through each set of selected features from each RFE algorithms
    X_train, X_test, y_train, y_test = split_scaler(i, dep_Y)
    
    r2 = Linear(X_train,y_train,X_test)
    r2_linear.append(r2)
    
    r2 = svm_linear(X_train,y_train,X_test)
    r2_svm_linear.append(r2)
    
    r2 = svm_NL(X_train,y_train,X_test)
    r2_svm_NL.append(r2)    
    
    r2 = Decision(X_train,y_train,X_test)
    r2_Decision.append(r2)
    
    r2 = random(X_train,y_train,X_test)
    r2_random.append(r2)
    
    

C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\utils\fixes.py:230: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if _joblib.__version__ >= LooseVersion('0.12'):
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\ensemble\base.py:158: DeprecationWarning: `np.int` is a deprecated alias for the builtin `int`. To silence this warning, use `int` by itself. Doing this will not modify any behavior and is safe. When replacing `np.int`, you may wish to use e.g. `np.int64` or `np.int32` to specify the precision. If you wish to review your current use, check the release note link for additional information.
Deprecated in NumPy 1.20; for more details and guidance: https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations
  dtype=np.int)
C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\utils\fixes.py:230: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if _joblib.__version__ >= LooseVersion('0.12'

In [41]:
result = rfe_regression(r2_linear, r2_svm_linear, r2_svm_NL, r2_Decision, r2_random)
result

,Linear,SVML,SVMNL,DT,RF
Linear,0.941886,-0.062807,-0.064146,0.879673,0.958281
SVM,0.941886,-0.062807,-0.064146,0.879673,0.958281
DT,0.932548,-0.062185,-0.0641,0.917671,0.955786
RF,0.932548,-0.062185,-0.0641,0.917671,0.955786


In [42]:
# #let's go with features selected by Liner and model created by RF

final_seleceted_feat = rfe_feature_list[0]
X_train, X_test, y_train, y_test = train_test_split(final_seleceted_feat, dep_Y, test_size=0.25, random_state=0)

# to standardise the data
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# since this is regression and o/p continuous numerical value, o/p should also be standardized.
# scy = StandardScaler()
# y_train = scy.fit_transform(y_train)
# y_test = scy.transform(y_test)

#save the sacled model for delpyment stage.
import joblib
joblib.dump(sc, "scaler_ip.pkl")
# joblib.dump(scy, "scaler_op.pkl")

from sklearn.ensemble import RandomForestRegressor
regressor = RandomForestRegressor(n_estimators = 10, random_state = 0)
regressor.fit(X_train, y_train)



C:\Anaconda3\envs\aiml\lib\site-packages\sklearn\utils\fixes.py:230: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if _joblib.__version__ >= LooseVersion('0.12'):


RandomForestRegressor(bootstrap=True, criterion='mse', max_depth=None,
                      max_features='auto', max_leaf_nodes=None,
                      min_impurity_decrease=0.0, min_impurity_split=None,
                      min_samples_leaf=1, min_samples_split=2,
                      min_weight_fraction_leaf=0.0, n_estimators=10,
                      n_jobs=None, oob_score=False, random_state=0, verbose=0,
                      warm_start=False)

In [45]:
import pickle
filename="finalized_model_rf.sav"
pickle.dump(regressor,open(filename, 'wb'))